# Evaluate on the 150 Daily Mirror sentences

Each hand-tagged Daily Mirror sentence is run through:
1. the **HMM tagger** (`hmm_tagger.pkl`);
2. the **PCFG parser in joint mode**: it parses the words and picks the POS tags itself (with its
   unknown-word signatures);
3. the **PCFG parser in pipeline mode**: it parses on top of the HMM's tags.

Gold parse trees don't exist for these sentences, so parsing performance is measured through the
POS tags each parse assigns (accuracy, per-tag precision / recall / F1, confusion matrix) plus
parse statistics (coverage, log-probability, time). Bracket-level PARSEVAL scores are in
`build_pcfg.ipynb`, on held-out treebank trees.

In [1]:
import sys, json, csv, time, pickle, math, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
from collections import Counter
from src.data_utils import load_dm_sentences, load_gold_tagged, load_clean_treebank, train_test_indices
from src.pcfg_parser import load_grammar, ViterbiCKYParser, debinarize, tree_pos
from src import evaluate as ev

with open('../results/treebank_parseval.json') as f:
    BEAM = json.load(f)['settings']['beam']
sentences = {int(r['id']): r for r in load_dm_sentences()}
gold = load_gold_tagged()
with open('../models/hmm_tagger.pkl', 'rb') as f:
    hmm_tagger = pickle.load(f)
parser = ViterbiCKYParser(load_grammar('../models/pcfg_grammar.txt'), beam=BEAM)

# the HMM's vocabulary = words in its training split
tb = load_clean_treebank(); train_idx, _ = train_test_indices(len(tb))
hmm_vocab = {w for i in train_idx for w, _ in tb[i]}
print(len(gold), 'gold sentences,', sum(map(len, gold.values())), 'tokens')

150 gold sentences, 2011 tokens


In [2]:
records = []
for sid, gold_sent in sorted(gold.items()):
    words = [w for w, _ in gold_sent]; gold_tags = [t for _, t in gold_sent]
    hmm = [t for _, t in hmm_tagger.tag(words)]
    t0 = time.time(); joint_tree, joint_lp = parser.parse(words); joint_time = time.time() - t0
    t0 = time.time(); pipe_tree, pipe_lp = parser.parse(words, hmm); pipe_time = time.time() - t0
    joint_tree = debinarize(joint_tree) if joint_tree else None
    pipe_tree = debinarize(pipe_tree) if pipe_tree else None
    records.append({'sid': sid, 'words': words, 'gold': gold_tags, 'hmm': hmm,
                    'pcfg': tree_pos(joint_tree) if joint_tree else hmm,
                    'joint_tree': joint_tree, 'joint_lp': joint_lp, 'joint_time': joint_time,
                    'pipe_tree': pipe_tree, 'pipe_lp': pipe_lp, 'pipe_time': pipe_time})
print('joint parses found:', sum(r['joint_tree'] is not None for r in records), '/', len(records))
print('pipeline parses found:', sum(r['pipe_tree'] is not None for r in records), '/', len(records))

joint parses found: 150 / 150
pipeline parses found: 144 / 150


## Overall tagging metrics

In [3]:
flat = lambda key: [t for r in records for t in r[key]]
words_all, gold_all = flat('words'), flat('gold')
systems = {'hmm_tagger': (flat('hmm'), lambda w: w in hmm_vocab),
           'pcfg_joint': (flat('pcfg'), parser.known)}
metrics = {}
for name, (pred, is_known) in systems.items():
    per_tag = ev.per_tag_scores(gold_all, pred)
    metrics[name] = {
        'token_accuracy': ev.accuracy(gold_all, pred),
        **ev.averaged_scores(per_tag),
        'sentence_exact_match': sum(r['gold'] == r['hmm' if name == 'hmm_tagger' else 'pcfg'] for r in records) / len(records),
        'known_vs_unknown': ev.known_unknown_accuracy(words_all, gold_all, pred, is_known),
        'top_confusions': [[g, p, c] for (g, p), c in ev.top_confusions(gold_all, pred)],
        'per_tag': per_tag,
    }
    m = metrics[name]
    print(f"{name:11s} accuracy={m['token_accuracy']:.4f} macro-P={m['macro_precision']:.3f} "
          f"macro-R={m['macro_recall']:.3f} macro-F1={m['macro_f1']:.3f} weighted-F1={m['weighted_f1']:.3f} "
          f"exact-sent={m['sentence_exact_match']:.3f}")
    for k, v in m['known_vs_unknown'].items():
        print(f"            {k:8s} words: accuracy={v['accuracy']:.3f} over {v['tokens']} tokens")

hmm_tagger  accuracy=0.8270 macro-P=0.744 macro-R=0.792 macro-F1=0.733 weighted-F1=0.830 exact-sent=0.133
            known    words: accuracy=0.939 over 1680 tokens
            unknown  words: accuracy=0.257 over 331 tokens
pcfg_joint  accuracy=0.9299 macro-P=0.896 macro-R=0.866 macro-F1=0.873 weighted-F1=0.930 exact-sent=0.420
            known    words: accuracy=0.956 over 1680 tokens
            unknown  words: accuracy=0.798 over 331 tokens


In [4]:
parse_stats = {}
for mode in ('joint', 'pipe'):
    ok = [r for r in records if r[f'{mode}_tree'] is not None]
    parse_stats['pcfg_joint' if mode == 'joint' else 'pcfg_hmm_pipeline'] = {
        'coverage': len(ok) / len(records),
        'mean_logprob': sum(r[f'{mode}_lp'] for r in ok) / len(ok),
        'mean_logprob_per_token': sum(r[f'{mode}_lp'] / len(r['words']) for r in ok) / len(ok),
        'mean_parse_seconds': sum(r[f'{mode}_time'] for r in records) / len(records),
    }
print(json.dumps(parse_stats, indent=2))

{
  "pcfg_joint": {
    "coverage": 1.0,
    "mean_logprob": -84.3987427875993,
    "mean_logprob_per_token": -6.301655955573907,
    "mean_parse_seconds": 0.049469048182169593
  },
  "pcfg_hmm_pipeline": {
    "coverage": 0.96,
    "mean_logprob": -119.15335482715497,
    "mean_logprob_per_token": -8.926908977715527,
    "mean_parse_seconds": 0.03158988157908122
  }
}


In [5]:
with open('../results/tagging_metrics.json', 'w') as f:
    json.dump({'test_set': {'sentences': len(records), 'tokens': len(gold_all)},
               'systems': metrics, 'parsing': parse_stats}, f, indent=2)

labels, matrix = ev.confusion_matrix(gold_all, flat('pcfg'))
ev.write_confusion_csv('../results/confusion_matrix.csv', labels, matrix)
hmm_labels, hmm_matrix = ev.confusion_matrix(gold_all, flat('hmm'))
ev.write_confusion_csv('../results/confusion_matrix_hmm.csv', hmm_labels, hmm_matrix)

with open('../results/tagging_predictions.tsv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, delimiter='\t', lineterminator='\n')
    w.writerow(['sent_id', 'token_id', 'word', 'gold', 'hmm', 'pcfg', 'hmm_known', 'pcfg_known'])
    for r in records:
        for i, (word, g, h, p) in enumerate(zip(r['words'], r['gold'], r['hmm'], r['pcfg']), 1):
            w.writerow([r['sid'], i, word, g, h, p, int(word in hmm_vocab), int(parser.known(word))])

## Per-sentence results

`results/parse_results.csv` has one row per sentence: tagging accuracy of each system, the number of
unknown words, whether a parse was found, its log-probability and the bracketed tree.

In [6]:
with open('../results/parse_results.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['sent_id', 'section', 'sentence', 'tokens', 'hmm_oov', 'pcfg_oov',
                'hmm_accuracy', 'pcfg_accuracy', 'pcfg_errors', 'joint_parsed', 'joint_logprob',
                'joint_seconds', 'pipeline_parsed', 'pipeline_logprob', 'joint_tree'])
    for r in records:
        n = len(r['words'])
        w.writerow([r['sid'], sentences[r['sid']]['section'], sentences[r['sid']]['sentence'], n,
                    sum(x not in hmm_vocab for x in r['words']), sum(not parser.known(x) for x in r['words']),
                    round(ev.accuracy(r['gold'], r['hmm']), 4), round(ev.accuracy(r['gold'], r['pcfg']), 4),
                    '; '.join(f"{x}:{g}->{p}" for x, g, p in zip(r['words'], r['gold'], r['pcfg']) if g != p),
                    int(r['joint_tree'] is not None), round(r['joint_lp'], 3) if r['joint_lp'] is not None else '',
                    round(r['joint_time'], 3), int(r['pipe_tree'] is not None),
                    round(r['pipe_lp'], 3) if r['pipe_lp'] is not None else '',
                    r['joint_tree'].pformat(margin=10**6) if r['joint_tree'] else ''])

with open('../results/parse_trees.txt', 'w', encoding='utf-8') as f:
    for r in records:
        f.write(f"#{r['sid']}  {sentences[r['sid']]['sentence']}\n")
        f.write(f"log P = {r['joint_lp']:.2f}\n" if r['joint_tree'] else 'NO PARSE\n')
        if r['joint_tree']:
            f.write(r['joint_tree'].pformat() + '\n')
        f.write('\n')

import pandas as pd
df = pd.read_csv('../results/parse_results.csv')
df[['sent_id', 'tokens', 'hmm_oov', 'pcfg_oov', 'hmm_accuracy', 'pcfg_accuracy', 'joint_parsed', 'joint_logprob']].describe().round(3)

,sent_id,tokens,hmm_oov,pcfg_oov,hmm_accuracy,pcfg_accuracy,joint_parsed,joint_logprob
count,150.000,150.000,150.000,150.000,150.000,150.000,150.0,150.000
mean,75.500,13.407,2.207,2.207,0.823,0.929,1.0,-84.399
std,43.445,3.019,1.607,1.607,0.131,0.080,0.0,20.913
min,1.000,7.000,0.000,0.000,0.375,0.615,1.0,-137.191
25%,38.250,11.000,1.000,1.000,0.778,0.889,1.0,-99.206
50%,75.500,14.000,2.000,2.000,0.846,0.938,1.0,-88.006
75%,112.750,16.000,3.000,3.000,0.907,1.000,1.0,-67.748
max,150.000,19.000,8.000,8.000,1.000,1.000,1.0,-37.173


## Figures

In [7]:
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.makedirs('../results/figures', exist_ok=True)
plt.rcParams.update({'figure.dpi': 130, 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
HMM_C, PCFG_C = '#9aa5b1', '#2f6db3'

# 1. per-sentence accuracy distribution
fig, ax = plt.subplots(figsize=(6.2, 3.2))
bins = [i / 20 for i in range(8, 21)]
ax.hist([df.hmm_accuracy, df.pcfg_accuracy], bins=bins, color=[HMM_C, PCFG_C], label=['HMM tagger', 'PCFG (joint)'])
ax.set_xlabel('tagging accuracy of the sentence'); ax.set_ylabel('sentences'); ax.legend(frameon=False)
ax.set_title('Per-sentence POS accuracy, 150 Daily Mirror sentences', loc='left')
fig.tight_layout(); fig.savefig('../results/figures/per_sentence_accuracy.png'); plt.show()

# 2. accuracy vs share of unknown words
fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.scatter(df.hmm_oov / df.tokens, df.hmm_accuracy, s=14, color=HMM_C, label='HMM tagger')
ax.scatter(df.pcfg_oov / df.tokens, df.pcfg_accuracy, s=14, color=PCFG_C, label='PCFG (joint)')
ax.set_xlabel('share of tokens unseen in training'); ax.set_ylabel('sentence accuracy'); ax.legend(frameon=False)
ax.set_title('Unknown words drive the errors', loc='left')
fig.tight_layout(); fig.savefig('../results/figures/accuracy_vs_oov.png'); plt.show()

# 3. per-tag F1 for the most frequent gold tags
support = Counter(gold_all)
top_tags = [t for t, _ in support.most_common(18)]
fig, ax = plt.subplots(figsize=(6.8, 3.4))
x = range(len(top_tags))
ax.bar([i - .2 for i in x], [metrics['hmm_tagger']['per_tag'][t]['f1'] for t in top_tags], .4, color=HMM_C, label='HMM tagger')
ax.bar([i + .2 for i in x], [metrics['pcfg_joint']['per_tag'][t]['f1'] for t in top_tags], .4, color=PCFG_C, label='PCFG (joint)')
ax.set_xticks(list(x)); ax.set_xticklabels([f'{t}\n{support[t]}' for t in top_tags], fontsize=7)
ax.set_ylabel('F1'); ax.set_ylim(0, 1.05); ax.legend(frameon=False, ncol=2, loc='lower center', bbox_to_anchor=(0.5, 1.0))
ax.set_title('Per-tag F1 (tag, gold count)', loc='left', pad=22)
fig.tight_layout(); fig.savefig('../results/figures/per_tag_f1.png'); plt.show()

# 4. confusion matrix of the PCFG, most frequent tags, row-normalised
idx = [labels.index(t) for t in top_tags if t in labels]
sub = [[matrix[i][j] for j in idx] for i in idx]
norm = [[v / max(1, sum(row)) for v in row] for row in sub]
fig, ax = plt.subplots(figsize=(5.6, 5.0))
im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
names = [labels[i] for i in idx]
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
for i, row in enumerate(sub):
    for j, v in enumerate(row):
        if v: ax.text(j, i, v, ha='center', va='center', fontsize=6, color='white' if norm[i][j] > .5 else 'black')
ax.set_xlabel('predicted'); ax.set_ylabel('gold'); ax.set_title('PCFG confusion matrix (counts, colour = row share)', loc='left')
fig.colorbar(im, fraction=.04); fig.tight_layout(); fig.savefig('../results/figures/confusion_matrix.png'); plt.show()

## Error analysis

In [8]:
for name in systems:
    print(name, 'most frequent confusions (gold -> predicted, count):')
    for g, p, c in metrics[name]['top_confusions'][:10]:
        print(f'   {g:5s} -> {p:5s} {c}')
worst = df.sort_values('pcfg_accuracy').head(8)
worst[['sent_id', 'pcfg_accuracy', 'hmm_accuracy', 'sentence', 'pcfg_errors']]

hmm_tagger most frequent confusions (gold -> predicted, count):
   NN    -> NNS   17
   VBD   -> VBN   16
   NNP   -> PRP   11
   JJ    -> NN    11
   NN    -> NNP   8
   JJ    -> DT    8
   NNP   -> NN    7
   NNP   -> JJ    6
   JJ    -> VBN   6
   NNP   -> NNS   6
pcfg_joint most frequent confusions (gold -> predicted, count):
   VBD   -> VBN   16
   JJ    -> NN    10
   NNS   -> VBZ   8
   JJ    -> NNP   8
   NN    -> JJ    7
   VBP   -> VB    5
   NN    -> NNP   4
   VBN   -> VBD   4
   IN    -> RB    3
   VB    -> VBP   3


,sent_id,pcfg_accuracy,hmm_accuracy,sentence,pcfg_errors
108,109,0.6154,0.4615,The corresponding Low Grown varieties sold wel...,corresponding:JJ->NN; varieties:NNS->VBZ; sold...
79,80,0.6364,0.6364,Integrated eKYC capabilities also support regi...,Integrated:VBN->NNP; eKYC:NNP->VBZ; capabiliti...
128,129,0.6667,0.5833,Investigators collected samples for chemical a...,Investigators:NNS->VBZ; collected:VBD->VBN; ch...
94,95,0.7333,0.7333,"But alas, in 2000 the bubble burst with so man...",alas:UH->NNS; burst:VBD->VBP; so:RB->IN; under...
68,69,0.7500,0.5000,The Ex-Estate offerings comprised of 0.7 M/Kgs.,Ex-Estate:NNP->JJ; comprised:VBD->VBN
80,81,0.7500,0.6250,"Petrol accounted for 1.14 percentage points, b...",Petrol:NN->NNP; accounted:VBD->VBN; bus:NN->JJ...
134,135,0.7778,0.7778,The report also incorporated chemical and fore...,incorporated:VBD->VBN; chemical:JJ->NN
123,124,0.7778,0.7778,"The OPs, in general, were firm.",OPs:NNS->NNP; firm:JJ->NN


In [9]:
# the best-scoring parse of one sentence
r = records[39]
print(' '.join(r['words'])); r['joint_tree'].pretty_print()

The central message is brutally simple : nature does not need us , but we need nature .
                                                        TOP                                                  
                                                         |                                                    
                                                         S                                                   
                _________________________________________|_________________________________________________   
               |                              |          S                     |   |       |               | 
               |                              |     _____|____                 |   |       |               |  
               S                              |    |          VP               |   |       S               | 
        _______|___________                   |    |      ____|________        |   |    ___|____           |  
       |                   V